# EDA

In [1]:
import numpy as numpy
import pandas as pd
import matplotlib.pyplot as plt
import os
import plotly.express as px
import plotly.graph_objects as go

os.chdir("..")
RAW_DATA = 'data/raw/transactions.txt'

In [2]:
df = pd.read_json(RAW_DATA, lines=True)
df.head()

,accountNumber,customerId,creditLimit,availableMoney,transactionDateTime,transactionAmount,merchantName,acqCountry,merchantCountryCode,posEntryMode,...,echoBuffer,currentBalance,merchantCity,merchantState,merchantZip,cardPresent,posOnPremises,recurringAuthInd,expirationDateKeyInMatch,isFraud
0,737265056,737265056,5000,5000.0,2016-08-13T14:27:32,98.55,Uber,US,US,02,...,,0.0,,,,False,,,False,False
1,737265056,737265056,5000,5000.0,2016-10-11T05:05:54,74.51,AMC #191138,US,US,09,...,,0.0,,,,True,,,False,False
2,737265056,737265056,5000,5000.0,2016-11-08T09:18:39,7.47,Play Store,US,US,09,...,,0.0,,,,False,,,False,False
3,737265056,737265056,5000,5000.0,2016-12-10T02:14:50,7.47,Play Store,US,US,09,...,,0.0,,,,False,,,False,False
4,830329091,830329091,5000,5000.0,2016-03-24T21:04:46,71.18,Tim Hortons #947751,US,US,02,...,,0.0,,,,True,,,False,False


In [3]:
df.describe()

,accountNumber,customerId,creditLimit,availableMoney,transactionAmount,cardCVV,enteredCVV,cardLast4Digits,currentBalance
count,7.863630e+05,7.863630e+05,786363.000000,786363.000000,786363.000000,786363.000000,786363.000000,786363.000000,786363.000000
mean,5.372326e+08,5.372326e+08,10759.464459,6250.725369,136.985791,544.467338,544.183857,4757.417799,4508.739089
std,2.554211e+08,2.554211e+08,11636.174890,8880.783989,147.725569,261.524220,261.551254,2996.583810,6457.442068
min,1.000881e+08,1.000881e+08,250.000000,-1005.630000,0.000000,100.000000,0.000000,0.000000,0.000000
25%,3.301333e+08,3.301333e+08,5000.000000,1077.420000,33.650000,310.000000,310.000000,2178.000000,689.910000
50%,5.074561e+08,5.074561e+08,7500.000000,3184.860000,87.900000,535.000000,535.000000,4733.000000,2451.760000
75%,7.676200e+08,7.676200e+08,15000.000000,7500.000000,191.480000,785.000000,785.000000,7338.000000,5291.095000
max,9.993896e+08,9.993896e+08,50000.000000,50000.000000,2011.540000,998.000000,998.000000,9998.000000,47498.810000


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 786363 entries, 0 to 786362
Data columns (total 29 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   accountNumber             786363 non-null  int64  
 1   customerId                786363 non-null  int64  
 2   creditLimit               786363 non-null  int64  
 3   availableMoney            786363 non-null  float64
 4   transactionDateTime       786363 non-null  str    
 5   transactionAmount         786363 non-null  float64
 6   merchantName              786363 non-null  str    
 7   acqCountry                786363 non-null  str    
 8   merchantCountryCode       786363 non-null  str    
 9   posEntryMode              786363 non-null  str    
 10  posConditionCode          786363 non-null  str    
 11  merchantCategoryCode      786363 non-null  str    
 12  currentExpDate            786363 non-null  str    
 13  accountOpenDate           786363 non-null  str    
 14 

In [9]:
df.columns

Index(['accountNumber', 'customerId', 'creditLimit', 'availableMoney',
       'transactionDateTime', 'transactionAmount', 'merchantName',
       'acqCountry', 'merchantCountryCode', 'posEntryMode', 'posConditionCode',
       'merchantCategoryCode', 'currentExpDate', 'accountOpenDate',
       'dateOfLastAddressChange', 'cardCVV', 'enteredCVV', 'cardLast4Digits',
       'transactionType', 'echoBuffer', 'currentBalance', 'merchantCity',
       'merchantState', 'merchantZip', 'cardPresent', 'posOnPremises',
       'recurringAuthInd', 'expirationDateKeyInMatch', 'isFraud'],
      dtype='str')

In [10]:
df['transactionAmount']

0          98.55
1          74.51
2           7.47
3           7.47
4          71.18
           ...  
786358    119.92
786359     18.89
786360     49.43
786361     49.89
786362     72.18
Name: transactionAmount, Length: 786363, dtype: float64

## 1. Распределение сумм транзакций по фроду (лог-шкала)

In [20]:
df["transactionDateTime"] = pd.to_datetime(df["transactionDateTime"])

fig_amount = px.histogram(
    df.sample(50000, random_state=42),
    x="transactionAmount",
    color="isFraud",
    nbins=100,
    barmode="overlay",
    histnorm="percent",
)
fig_amount.show()

## 2. Доля фрода по месяцам

In [ ]:
monthly = (
    df
    .set_index("transactionDateTime")
    .groupby(pd.Grouper(freq="ME"))
    .agg(txn_count=("isFraud", "size"),
         fraud_rate=("isFraud", "mean"))
    .reset_index()
)

fig_monthly = px.line(
    monthly,
    x="transactionDateTime",
    y="fraud_rate",
    markers=True,
)
fig_monthly.update_layout(
    title="Доля мошеннических транзакций по месяцам",
)
fig_monthly.update_xaxes(title_text="Месяц")
fig_monthly.update_yaxes(title_text="Доля фрода")
fig_monthly.show()

## 3. Топ‑10 категорий мерчантов: объём и доля фрода

In [21]:
cat = (
    df
    .groupby("merchantCategoryCode")
    .agg(txn_count=("isFraud", "size"),
         fraud_rate=("isFraud", "mean"))
    .sort_values("txn_count", ascending=False)
    .head(10)
    .reset_index()
)

fig_cat = px.bar(
    cat,
    x="merchantCategoryCode",
    y="txn_count",
    color="fraud_rate",
    color_continuous_scale="Reds",
)
fig_cat.update_layout(
    title="ТОП‑10 категорий мерчантов по числу транзакций",
    coloraxis_colorbar_title="Доля фрода",
)
fig_cat.update_xaxes(title_text="Категория мерчанта")
fig_cat.update_yaxes(title_text="Кол-во транзакций")
fig_cat.show()

In [22]:
## 4. Связь суммы, фрода и наличия карты


In [24]:
fig_scatter = px.scatter(
    df.sample(min(len(df), 20000), random_state=42),  # чтобы не было слишком тяжело
    x="transactionAmount",
    y="availableMoney",
    color="isFraud",
    symbol="cardPresent",
    marginal_x="histogram",
    marginal_y="histogram",
)
fig_scatter.update_layout(
    title="Сумма vs доступные средства (цвет — фрод, символ — cardPresent)",
)
fig_scatter.update_xaxes(title_text="Сумма транзакции", type="log")
fig_scatter.update_yaxes(title_text="Доступные средства")
fig_scatter.show()